In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
words = open("../data.txt", "r").read().splitlines()

In [ ]:
stoi = {chr(i+97): i+1 for i in range(26)}
stoi['.'] = 0

itos = {value : key for key, value in stoi.items()}

vocabsize = 27

In [ ]:
block_size = 3

def build_dataset(data):
    X, Y = [], []

    for w in data:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)

            context = context[1:] + [ix]

    return torch.tensor(X), torch.tensor(Y)

import random
random.seed(42)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtrain, ytrain = build_dataset(words[:n1])
Xval, yval = build_dataset(words[n1:n2])
Xtest, ytest = build_dataset(words[n2:])

In [ ]:
# utility function to be used later when comparing manual grads with pytorch grads later
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [ ]:
nembedd = 10
nhidden = 64

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocabsize, nembedd), generator=g)
W1 = torch.randn(((nembedd*block_size), nhidden), generator=g) * (5/3) * ((nembedd*block_size) ** (-0.5))   # Kaiming init
b1 = torch.randn(nhidden, generator=g) * 0.1
W2 = torch.randn((nhidden, vocabsize), generator=g) * 0.1
b2 = torch.randn(vocabsize, generator=g) * 0.1

bngain = torch.randn((1, nhidden)) * 0.1 + 1.0
bnbias = torch.randn((1, nhidden)) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

In [ ]:
batchsize = 32
n = batchsize

ix = torch.randint(0, Xtrain.shape[0], (batchsize, ), generator=g)
Xb, yb = Xtrain[ix], ytrain[ix]

Forward Pass

In [ ]:
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)

# Linear layer
hprebn = embcat @ W1 + b1

#BatchNorm layer
bnmean = 1/n* hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmean
bndiff2 = bndiff ** 2
bnvar = 1/(n-1) * bndiff2.sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff * bnvar_inv
hpreact = bnraw * bngain + bnbias

# Activation layer
h = torch.tanh(hpreact)

# Linear layer 2
logits = h @ W2 + b2

# Cross Entropy loss
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1   # softmax

probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), yb].mean()

# PyTorch backward pass
for p in parameters:
  p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
         bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmean,
         embcat, emb]:
  t.retain_grad()
loss.backward()
loss


### Manual Backprop

In [ ]:
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), yb] = -1/n

dprobs = (1/probs)*dlogprobs
dcounts_sum_inv = (counts*dprobs).sum(1, keepdim=True)
dcounts_sum = -dcounts_sum_inv * (counts_sum**-2)
dcounts = dprobs*counts_sum_inv + dcounts_sum*torch.ones_like(counts)
dnorm_logits = dcounts * counts
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)

dlogits = dnorm_logits + F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

In [ ]:
dh = dlogits @ W2.T
"""
h = [[a11, a12], [a21, a22]]
W2 = [[b11, b12], [b21, b22]]

logits = h @ W2 = [[h11, h12],[h21, h22]]

h11 = a11*b11 + a12*b21 + c11
h12 = a11*b12 + a12*b22 + c12 and so on

dlogits/da11 = b11*(dL/dh11) + b12*(dL/dh12) and so on

this condenses to dlogits/dh @ W2.Transpose
"""

dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

In [ ]:
dhpreact = (1 - h**2) * dh
dbngain = (dhpreact * bnraw).sum(0, keepdim = True)
dbnraw = dhpreact * bngain
dbnbias = dhpreact.sum(0, keepdim=True)
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbndiff = bnvar_inv * dbnraw # not complete grad yet as it also appears in bndiff2
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
dbndiff2 = (1.0/(n-1))*torch.ones_like(bndiff2) * dbnvar
dbndiff += dbndiff2 * 2 * bndiff
dbnmean = -dbndiff.sum(0, keepdim=True)
dhprebn = dbndiff + dbnmean * (1/n * torch.ones_like(hprebn))


In [ ]:
dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0, keepdim=True)

demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for i in range(emb.shape[0]):
    for j in range(emb.shape[1]):
        dC[Xb[i][j]] += demb[i][j]

In [ ]:
demb.shape, emb.shape, dC.shape, C.shape

### Comparison

In [ ]:
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnbias', dbnbias, bnbias)
cmp('bnraw', dbnraw, bnraw)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bndiff', dbndiff, bndiff)
cmp('bnmean', dbnmean, bnmean)
cmp('hprebn', dhprebn, hprebn)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

In [ ]:
# forward pass

# before:
# logit_maxes = logits.max(1, keepdim=True).values
# norm_logits = logits - logit_maxes # subtract max for numerical stability
# counts = norm_logits.exp()
# counts_sum = counts.sum(1, keepdims=True)
# counts_sum_inv = counts_sum**-1 # if I use (1.0 / counts_sum) instead then I can't get backprop to be bit exact...
# probs = counts * counts_sum_inv
# logprobs = probs.log()
# loss = -logprobs[range(n), Yb].mean()

# now:
loss_fast = F.cross_entropy(logits, yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

In [ ]:
# backward pass

dlogits = F.softmax(logits, 1)
dlogits[range(n), yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits)

In [ ]:
#logprobs.grad[1], dlogprobs[0], logprobs.grad[0] == dlogprobs[0]

In [ ]:
F.softmax(logits, 1)[0]

In [ ]:
dlogits[0] * n

In [ ]:
dlogits[0].sum()

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(dlogits.detach(), cmap='gray')

In [ ]:
# forward pass

# before:
# bnmeani = 1/n*hprebn.sum(0, keepdim=True)
# bndiff = hprebn - bnmeani
# bndiff2 = bndiff**2
# bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n-1, not n)
# bnvar_inv = (bnvar + 1e-5)**-0.5
# bnraw = bndiff * bnvar_inv
# hpreact = bngain * bnraw + bnbias

# now:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

In [ ]:
dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))

cmp('hprebn', dhprebn, hprebn)

In [ ]:
dhprebn.shape, bngain.shape, bnvar_inv.shape, dbnraw.shape, dbnraw.sum(0).shape

### Putting it all together

In [ ]:
# init
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 200 # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647) # for reproducibility
C  = torch.randn((vocabsize, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1
# Layer 2
W2 = torch.randn((n_hidden, vocabsize),          generator=g) * 0.1
b2 = torch.randn(vocabsize,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters)) # number of parameters in total
for p in parameters:
  p.requires_grad = True

# same optimization as last time
max_steps = 200000
batch_size = 32
n = batch_size # convenience
lossi = []

with torch.no_grad():

  for i in range(max_steps):

      # minibatch construct
      ix = torch.randint(0, Xtrain.shape[0], (batch_size,), generator=g)
      Xb, Yb = Xtrain[ix], ytrain[ix] # batch X,Y

      # forward pass
      emb = C[Xb] # embed the characters into vectors
      embcat = emb.view(emb.shape[0], -1) # concatenate the vectors
      # Linear layer
      hprebn = embcat @ W1 + b1 # hidden layer pre-activation
      # BatchNorm layer
      bnmean = hprebn.mean(0, keepdim=True)
      bnvar = hprebn.var(0, keepdim=True, unbiased=True)
      bnvar_inv = (bnvar + 1e-5)**-0.5
      bnraw = (hprebn - bnmean) * bnvar_inv
      hpreact = bngain * bnraw + bnbias
      # Non-linearity
      h = torch.tanh(hpreact) # hidden layer
      logits = h @ W2 + b2 # output layer
      loss = F.cross_entropy(logits, Yb) # loss function

      # backward pass
      for p in parameters:
        p.grad = None
      #loss.backward() # for correctness comparisons

      # manual backprop
      dlogits = F.softmax(logits, 1)
      dlogits[range(n), Yb] -= 1
      dlogits /= n
      # 2nd layer backprop
      dh = dlogits @ W2.T
      dW2 = h.T @ dlogits
      db2 = dlogits.sum(0)
      # tanh
      dhpreact = (1.0 - h**2) * dh
      # batchnorm backprop
      dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
      dbnbias = dhpreact.sum(0, keepdim=True)
      dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))
      # 1st layer
      dembcat = dhprebn @ W1.T
      dW1 = embcat.T @ dhprebn
      db1 = dhprebn.sum(0)
      # embedding
      demb = dembcat.view(emb.shape)
      dC = torch.zeros_like(C)
      for k in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
          ix = Xb[k,j]
          dC[ix] += demb[k,j]
      grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

      # update
      lr = 0.1 if i < 100000 else 0.01 # step learning rate decay
      for p, grad in zip(parameters, grads):
        p.data += -lr * grad

      # track stats
      if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
      lossi.append(loss.log10().item())


In [ ]:
# useful for checking gradients
# for p,g in zip(parameters, grads):
#   cmp(str(tuple(p.shape)), g, p)

In [ ]:
# calibrate the batch norm at the end of training

with torch.no_grad():
  # pass the training set through
  emb = C[Xtrain]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  # measure the mean/std over the entire training set
  bnmean = hpreact.mean(0, keepdim=True)
  bnvar = hpreact.var(0, keepdim=True, unbiased=True)


In [ ]:
# evaluate train and val loss

@torch.no_grad() # this decorator disables gradient tracking
def split_loss(split):
  x,y = {
    'train': (Xtrain, ytrain),
    'val': (Xval, yval),
    'test': (Xtest, ytest),
  }[split]
  emb = C[x] # (N, block_size, n_embd)
  embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
  hpreact = embcat @ W1 + b1
  hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
  h = torch.tanh(hpreact) # (N, n_hidden)
  logits = h @ W2 + b2 # (N, vocab_size)
  loss = F.cross_entropy(logits, y)
  print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
# sample from the model
g = torch.Generator().manual_seed(2147483647 + 10)

for _ in range(20):
    
    out = []
    context = [0] * block_size # initialize with all ...
    while True:
      # ------------
      # forward pass:
      # Embedding
      emb = C[torch.tensor([context])] # (1,block_size,d)      
      embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
      h = torch.tanh(hpreact) # (N, n_hidden)
      logits = h @ W2 + b2 # (N, vocab_size)
      # ------------
      # Sample
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))